In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm  # EfficientNetV2 ve CvT gibi modeller için kritik
import os
import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- UYARILAR VE OPTİMİZASYON İÃƒâ€¡İN (İsteğe Bağlı) ---
# AdamW ve LR Scheduler'ları için ek modül (CvT ve EfficientNetV2 için önemlidir)
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau

# Uyarıları gizlemek için
import warnings
warnings.filterwarnings("ignore")

# --- GLOBAL AYARLAR ---
# Random Seed (Tekrarlanabilir sonuçlar için)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# --- KONFİGÜRASYON (EfficientNetV2) ---

# EfficientNetV2 S (Small) Modeli (timm kodu: efficientnetv2_s)
MODEL_NAME = 'tf_efficientnetv2_m'

# Deney Adı
EXPERIMENT_NAME = "EfficientNetV2_Medium_Baseline_MediumLarge_Run1"

# Hiperparametreler
# BATCH_SIZE'ı GPU belleğinde yer açmak için 64'ten 32'ye düşürelim.
BATCH_SIZE = 24
EPOCHS = 50
LEARNING_RATE = 0.0003
NUM_CLASSES = 8
# Modelin kendi yapısında Dropout olduğundan, transfer öğrenme yaparken bu değeri düşük tutmak yeterli.
DROPOUT_RATE = 0.2

# --- DOSYA YOLLARI ---
DATA_DIR = "../data/prepared-data"

# Sonuçların kaydedileceği yer
OUTPUT_DIR = f"../models/pytorch/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cihaz Kontrolü
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Kayıt Yeri: {OUTPUT_DIR}")

In [ ]:
# ImageNet Normalize Değerleri
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Veri Dönüşümleri
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
}

# Datasetleri Oluştur
image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

# DataLoaderları Oluştur
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE,
                             shuffle=(x=='train'), num_workers=4, pin_memory=True)
               for x in ['train', 'val', 'test']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

print(f"Sınıflar: {class_names}")
print(f"Eğitim Verisi: {dataset_sizes['train']}")
print(f"Validasyon Verisi: {dataset_sizes['val']}")

In [ ]:
def create_model(model_name: str, num_classes: int, dropout_rate: float, device: str):
    print(f"Model indiriliyor: {model_name}...")

    # pretrained=True ile ImageNet ağırlıklarını alıyoruz
    try:
        model = timm.create_model(model_name,
                                  pretrained=True,
                                  num_classes=num_classes,
                                  drop_rate=dropout_rate)
    except Exception as e:
        print(f"Hata: Model '{model_name}' yüklenemedi. Kontrol edin. Hata: {e}")
        return None

    model = model.to(device)
    return model
model = create_model(MODEL_NAME, NUM_CLASSES, DROPOUT_RATE, DEVICE)
criterion = nn.CrossEntropyLoss()
weight_decay = 1e-4 if 'efficientnet' in MODEL_NAME else 5e-2 # CvT için 5e-2 daha agresif.

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=weight_decay)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("Model GPU'ya yüklendi ve eğitime hazır.")

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    required_globals = ["dataloaders", "dataset_sizes", "OUTPUT_DIR", "DEVICE"]    missing = [name for name in required_globals if name not in globals()]    if missing:        raise RuntimeError(            f"Eksik degisken(ler): {missing}. Lutfen once veri ve konfigürasyon hucrelerini calistirin."        )    since = time.time()
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}


    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Batch Döngüsü
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    # Tekrar kontrol: outputs, (N, C) boyutunda olmalıdır.
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Epoch Sonucu Hesaplama
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Geçmişi Kaydet
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            # --- SCHEDULER VE MODEL KAYDI DÜZENLEMELERİ ---

            if phase == 'train':
                # CosineAnnealingLR: Her eğitim epoch'undan sonra adım atmalıdır.
                # (ReduceLROnPlateau gibi val kaybına bağlı değildir)
                # Sadece eğitim fazında güncellenir.
                if isinstance(scheduler, (torch.optim.lr_scheduler.CosineAnnealingLR, torch.optim.lr_scheduler.OneCycleLR)):
                    scheduler.step()
                    # Güncel LR'ı görmek faydalı olabilir
                    current_lr = optimizer.param_groups[0]['lr']
                    print(f"Current LR: {current_lr:.6f}")

            if phase == 'val':
                # ReduceLROnPlateau kullanıyorsanız, burayı tekrar açın:
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(epoch_loss)

                # En iyi modeli kaydet (Validasyon başarısına göre)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    save_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
                    torch.save(model.state_dict(), save_path)
                    print(f"En İyi Model! ({best_acc:.4f}) -> Kaydedildi ve Yolu: {save_path}")

    time_elapsed = time.time() - since
    print(f'\nEğitim Tamamlandı: {time_elapsed // 60:.0f}dk {time_elapsed % 60:.0f}sn')
    print(f'En İyi Validasyon Doğruluğu: {best_acc:.4f}')

    # En iyi ağırlıkları geri yükle
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth')))
    return model, history